<div align="center">

# NOKTA Atölyesi: Yapay Zekâ ile Çevresel Veri Analizi ve Yorumlama

**İki saatlik atölyenin Colab defteri** · NOKTA, Nesnel Ölçüm, Katılım, Takip ve Aksiyon Ağı · ÜNİDES 2026

**Prof. Dr. Utku Köse**<br>
Süleyman Demirel Üniversitesi, Isparta · Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM)<br>
University of North Dakota · Universidad Panamericana · Vel Tech University

[Atölye sitesi](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/) · [GitHub](https://github.com/utkukose/env-data-ai-nokta2026-NB-workshop) · [ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com)

*Bu atölye alandaki güncel gelişmelere göre güncellenmektedir. İlk sürüm: Kasım 2026.*

</div>

## Bu defter nasıl kullanılır

Önce Drive'ınıza bir kopya kaydedin (Dosya, Drive'a kopya kaydet). Böylece yaptığınız değişiklikler korunur. Ardından hücreleri Shift+Enter ile sırayla çalıştırın, çünkü sonraki hücreler önceki hücrelerin sonuçlarını kullanmaktadır. Defterin her kısmı, [atölye sitesinde](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/) aynı numarayı taşıyan bölüme karşılık gelir: Defterin 1. kısmı sitenin 1. Bölümüne, 5. kısmı da sitenin 5. Bölümüne aittir. 0. kısım veriyi yükler. Colab'da herhangi bir kurulum gerekmez.

Dört tür hücre çalışmaya yol gösterir. **Python notu** hücreleri Python dilinin bir parçasını ilk kullanıldığı yerde açıklar. **Alıştırmalar** tek bir sayı ya da ad ister ve cevabı `check(...)` ile hemen denetler. Örnek çözümleri katlanmış hücrelerdedir ve ancak gerçekten denedikten sonra açılmalıdır. **Gözlemle ve düşün** hücreleri bir çıktı hakkında açık uçlu sorular sorar. Bu hücrelerin çözümü yoktur, çünkü cevap sizin kendi akıl yürütmenize bağlıdır. Veriler sentetiktir ve NOKTA saha protokolünü izler. 0. kısım bunun yerine gerçek NOKTA dosyalarının nasıl yükleneceğini gösterir.

## 0. Kurulum ve veri

İlk hücre kütüphaneleri yükler, altı nokta türünün sırasını, renklerini ve adlarını tanımlar. Alıştırmaları puanlayan `check` yardımcısı da bu hücrededir. İkinci hücre alıştırmaların beklenen cevaplarını katlanmış bir biçimde tutar. Üçüncü hücre dört dosya okur: Ölçümler (her 10 saniyede bir satır), saha formları (her ziyaret için bir satır), üç istasyonun yan yana testi ve yalnızca sentetik veride bulunan, bilerek yerleştirilmiş sorunların listesi. Bir dosya defterin yanında bulunamazsa GitHub'dan indirilir.

**Kendi NOKTA veriniz.** `DATA_SOURCE = "own"` yapın ve `data/nokta_template.csv` sütunlarına sahip CSV dosyalarınızı soldaki Dosyalar paneliyle yükleyin. Dosya adları `OWN_FILES` sözlüğünde belirtilir. Kayıt cihazınız başka sütun adları kullanıyorsa `COLUMN_MAP` sözlüğünü doldurun, örneğin `{"Temp": "temperature_c"}`. Bu hücreden sonraki her şey sizin verinizle çalışır.

In [ ]:
import json, re, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RANDOM_STATE = 2026
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 140)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:                      # yerine matplotlib kullanılır
    HAS_PLOTLY = False

TYPE_ORDER = ["shaded", "traffic", "reference", "sunny_hard", "crowded", "green"]
TYPE_COLOR = {"shaded": "#2a78d6", "traffic": "#eb6834", "reference": "#1baf7a",
              "sunny_hard": "#eda100", "crowded": "#e87ba4", "green": "#008300"}
TYPE_NAME = {"shaded": "Gölgeli", "traffic": "Trafik",
             "reference": "Referans", "sunny_hard": "Güneşli sert",
             "crowded": "Yoğun", "green": "Yeşil"}


def check(name, value, tol=0.01):
    """Bir alıştırma cevabını beklenen cevapla karşılaştırır."""
    exp = EXPECTED.get(name)
    if value is None:
        print(f"{name}: henüz cevap yok")
        return False
    if isinstance(exp, str):
        ok = str(value).strip().lower() == exp
    else:
        ok = abs(float(value) - exp) <= tol
    print(f"{name}: " + ("doğru ✓" if ok else "henüz değil, yeniden deneyin"))
    return ok

print("Hazır. 3B grafikler için Plotly:", HAS_PLOTLY)

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; ancak alıştırmaları denedikten sonra açın)
EXPECTED = {
    'Alıştırma 1.A': 139.4529,
    'Alıştırma 2.A': -1.105,
    'Alıştırma 2.B': 9.2611,
    'Alıştırma 3.A': 5,
    'Alıştırma 4.A': 'vehicles_per_min',
    'Alıştırma 5.A': 4,
}

In [ ]:
DATA_SOURCE = "synthetic"        # "synthetic" ya da "own"
REPO_RAW = ("https://raw.githubusercontent.com/utkukose/"
            "env-data-ai-nokta2026-NB-workshop/main/data/")
FILES = {"readings": "nokta_synthetic.csv",
         "forms": "nokta_field_forms_synthetic.csv",
         "coloc": "nokta_colocation_synthetic.csv",
         "truth": "nokta_synthetic_qc_truth.csv"}
OWN_FILES = {"readings": "my_readings.csv", "forms": "my_field_forms.csv",
             "coloc": "my_colocation.csv"}
COLUMN_MAP = {}                    # ör. {"Temp": "temperature_c"}


def fetch(name):
    """Varsa yerel kopya, yoksa GitHub'dan indirme."""
    for folder in (Path("data"), Path("../data"), Path("workshop_data")):
        if (folder / name).exists():
            return folder / name
    Path("workshop_data").mkdir(exist_ok=True)
    target = Path("workshop_data") / name
    urllib.request.urlretrieve(REPO_RAW + name, target)
    return target


if DATA_SOURCE == "own":
    paths = {k: Path(v) for k, v in OWN_FILES.items()}
else:
    paths = {k: fetch(v) for k, v in FILES.items()}
print({k: str(v) for k, v in paths.items()})

## Python adımı: Kodu okumak için beş fikir

Python bilmeyen katılımcılar bu adımı beş dakikada okuyabilir ve canlı oturumda atlayabilir.

1. **Değişkenler** bir değeri bir adla saklar: `rh = 43.0`.
2. **Listeler** birkaç değeri sırayla tutar: `pm = [14.2, 15.0, 13.8]` ve `pm[0]` ilk değerdir.
3. **Sözlükler** adları değerlere bağlar: `{"P1": "shaded"}`.
4. **Fonksiyonlar** bir hesabı paketler: `def dew_point(t, rh): ...` tanımı `dew_point(13.3, 43)` ile çalıştırılır.
5. **pandas tabloları (DataFrame)** NOKTA ölçümlerini tutar: Her ölçüm için bir satır, her kanal için bir sütun. `df["pm25_ugm3"]` bir sütundur, `df.groupby("point_type")` satırları noktalara göre ayırır.

In [ ]:
# Python alıştırması: Dili öğrenmek için küçük bir örnek
pm = [14.2, 15.0, 13.8, 14.6, 97.3, 15.1, 14.4]      # PM2,5 ölçümlerinden bir liste
print("ölçüm sayısı:", len(pm))
print("ortalama:", round(sum(pm) / len(pm), 2))
print("medyan:", sorted(pm)[len(pm) // 2])
point = {"id": "P2", "type": "traffic", "vehicles_15min": 104}
print(f"{point['id']}: {point['type']},",
      point["vehicles_15min"], "araç, 15 dakikada")

**Python notu.** `f"..."` biçimlendirilmiş bir metindir: `{ }` içindeki değerler metne yazılır. `round(x, 2)` iki ondalık basamak bırakır, `sorted(...)` sıralanmış bir kopya döndürür, `//` ise kalansız böler. Bu yüzden `sorted(pm)[len(pm) // 2]` sıralanmış listenin ortadaki değeridir. Çıktı, atölye sitesinin 2. Bölümündeki konuyu şimdiden göstermektedir: Tek bir 97,3 ölçümü ortalamayı 26,34'e çıkarır, medyan ise 14,6'da kalır.

## 1. Ölç: Veri olarak kampüs

*Atölye sitesi: [1. Bölüm](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/part1.html)*

Defterin 1. kısmı ham ölçümleri, 2-4. kısımların kullandığı temiz dakika tablosuna dönüştürür. Düşük maliyetli sensörlerin tipik hatalarını işaretler, üç istasyon arasındaki sapmaları giderir, partikül ölçümlerini neme göre düzeltir ve son olarak her ziyareti dakika dakika özetler.

### 1.1 Yükle ve bak

`load_readings` fonksiyonu ölçüm dosyasını okur, zaman damgalarını tarihe çevirir ve her ziyaret için bir etiket oluşturur. Örneğin `A-R1-P2` etiketi Kampüs A, 1. tur ve P2 noktasını gösterir. Hücre ardından saha formlarını ve yan yana testi yükler, ölçüm, ziyaret ve kampüs sayılarını yazdırır. Sondaki küçük tablo her kampüs ve turda ölçülen nokta sayısını verir. Bu sayı her yerde altı olmalıdır.

In [ ]:
def load_readings(path):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df["visit"] = (df["campus"] + "-R" + df["round"].astype(str) + "-"
                   + df["point_id"])
    return df


CHANNELS = ['temperature_c', 'rel_humidity_pct', 'pressure_hpa', 'pm1_ugm3',
    'pm25_ugm3', 'pm10_ugm3', 'co2_ppm', 'light_lux', 'uva_wm2', 'uvb_wm2']
PM = ['pm1_ugm3', 'pm25_ugm3', 'pm10_ugm3']
LUX_MAX = 65535

df = load_readings(paths["readings"])
df = df.rename(columns=COLUMN_MAP)
forms = pd.read_csv(paths["forms"])
coloc = pd.read_csv(paths["coloc"])
print(df.shape[0], "ölçüm,", forms.shape[0], "ziyaret,",
      df["campus"].nunique(), "kampüs")
display(df.head(3))
display(df.groupby(["campus", "round"])["point_id"].nunique().unstack())

**Python notu.** `df.groupby([...])[col].nunique()` her gruptaki farklı değerleri sayar, `.unstack()` ise iç grubu sütunlara çevirir.

Sonraki hücre tek bir ziyareti çizer: Kampüs A'da 1. turdaki trafiğe yakın nokta. Üç panel sıcaklığı, PM2,5 değerini ve CO₂'yi istasyonun varışından sonraki dakikalara göre gösterir. Turuncu bant, sensörlerin yeni yere henüz uyum sağladığı ilk iki dakikayı işaretler. PM2,5 ve CO₂'deki kısa tepeler, geçen araçların bıraktığı bulutlardır.

In [ ]:
visit = df[df["visit"] == "A-R1-P2"]     # Kampüs A, 1. tur, trafiğe yakın
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, ch in zip(axes, ["temperature_c", "pm25_ugm3", "co2_ppm"]):
    ax.plot(visit["seconds_since_arrival"] / 60, visit[ch], lw=1.4, color="#0E7A5F")
    ax.axvspan(0, 2, color="#eda100", alpha=0.15)
    ax.set_title(ch); ax.set_xlabel("varıştan sonraki dakika")
plt.tight_layout(); plt.show()

### 1.2 Kalite işaretleri

Ham veri asla değiştirilmez. `flag_quality` fonksiyonu bunun yerine her sorun için doğru ya da yanlış değerli bir sütun ekler: Taşımadan sonraki ilk 120 saniyedeki ısınma, kayıt boşlukları, 65.535 lx olan 16 bitlik sınırına ulaşmış ışık sensörü, %85'in üzerindeki bağıl nem, tek kanalda görülen partikül hataları, 300-2.000 ppm aralığının dışındaki CO₂ değerleri ve değeri art arda 12 ölçüm boyunca hiç değişmeyen donmuş bir CO₂ sensörü. Hata testi, atölye sitesinin 1. Bölümünde anlatılan değiştirilmiş z-skorunu kullanır [1]:

$$M_i = \frac{0{,}6745\,(x_i - \tilde{x})}{\mathrm{MAD}}, \qquad \mathrm{MAD} = \operatorname{median}_i \lvert x_i - \tilde{x} \rvert$$

Burada x<sub>i</sub> tek bir ölçüm, x̃ ölçümlerin medyanı, MAD ise medyandan mutlak sapmaların medyanıdır. 0,6745 çarpanı, veri normal dağıldığında M değerini olağan bir z-skoruyla karşılaştırılabilir kılar. Medyan ve MAD birkaç uç değerden neredeyse etkilenmez. Bu yüzden tek bir sıçrama, yayılımı büyüterek kendini gizleyemez.

*Formüllerden koda: `robust_z` M değerini hesaplar. `spike_scores` bunu her ölçümün beş ölçümlük kayan medyandan uzaklığına uygular. Böylece bir ziyaret boyunca görülen yavaş değişimler sıçrama sayılmaz. Bir partikül ölçümü üç koşul birlikte sağlandığında işaretlenir: |M| > 6 olmalı, ölçüm kayan medyanın iki katından büyük olmalı ve üç partikül boyutundan yalnızca biri sıçramalıdır. Son koşulun nedeni, gerçek bir bulutun bütün boyutları birlikte yükseltmesidir.*

In [ ]:
def robust_z(x):
    """Değiştirilmiş z-skoru: 0,6745 (x - medyan) / MAD (Leys vd., 2013)."""
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad == 0:
        return np.zeros_like(x)
    return 0.6745 * (x - med) / mad

def spike_scores(series, window=5):
    """Kayan medyandan uzaklığın dayanıklı z-skoru."""
    run_med = series.rolling(window, center=True, min_periods=1).median()
    return pd.Series(robust_z(series - run_med), index=series.index)

def flag_quality(df, z_limit=6.0, warmup_s=120, flat_len=12):
    """Her kalite sorunu için bir mantıksal sütun döndürür (ham veri korunur)."""
    f = pd.DataFrame(index=df.index)
    f["warmup"] = df["seconds_since_arrival"] < warmup_s
    f["gap"] = df[CHANNELS].isna().any(axis=1)
    f["saturated_light"] = df["light_lux"] >= LUX_MAX
    f["high_rh"] = df["rel_humidity_pct"] > 85
    jump = {}
    for ch in PM:
        z = df.groupby("visit")[ch].transform(spike_scores)
        run_med = df.groupby("visit")[ch].transform(
            lambda s: s.rolling(5, center=True, min_periods=1).median())
        jump[ch] = (z.abs() > z_limit) & (df[ch] > 2 * run_med)
    jump = pd.DataFrame(jump)
    for ch in PM:
        # gerçek bir bulut bütün partikül boyutlarını yükseltir, hata yalnızca birini
        f["spike_" + ch] = jump[ch] & ~jump.drop(columns=ch).any(axis=1)
    f["spike_co2_ppm"] = (df["co2_ppm"] <= 300) | (df["co2_ppm"] >= 2000)
    roll_sd = df.groupby("visit")["co2_ppm"].transform(
        lambda s: s.rolling(flat_len, min_periods=flat_len).std())
    end_of_run = (roll_sd == 0).astype(int)
    # işareti donmuş pencerenin tamamına geri yay
    spread = end_of_run[::-1].groupby(df["visit"][::-1]).transform(
        lambda s: s.rolling(flat_len, min_periods=1).max())[::-1]
    f["flatline_co2"] = spread.fillna(0).astype(bool)
    return f

def clean_readings(df, flags):
    """İşaretli değerleri eksik yapılmış (asla sıfır değil) veri kopyası."""
    out = df.copy()
    out.loc[flags["warmup"], CHANNELS] = np.nan
    out.loc[flags["saturated_light"], "light_lux"] = np.nan
    out.loc[flags["flatline_co2"] | flags["spike_co2_ppm"], "co2_ppm"] = np.nan
    for ch in PM:
        out.loc[flags["spike_" + ch], ch] = np.nan
    return out


flags = flag_quality(df)
display(flags.sum().rename("işaretli ölçüm").to_frame().T)

Sonraki hücre işaretleri sentetik veriye yerleştirilmiş sorunlarla karşılaştırarak puanlar. Her sorun için `planted` o sorunu taşıyan ölçümleri, `flagged` kuralın işaretlediği ölçümleri, `caught` ise bunlar arasındaki yerleştirilmiş ölçümleri sayar. İki PM2,5 hatası kaçırılır, çünkü gerçek bulutların içine düşerler ve orada tek kanaldaki bir sıçramayı gerçek kirlilikten ayırmak zordur. Grafik ardından Kampüs B'nin 1. turundaki ilk ziyarette CO₂ değerlerini gösterir. Donmuş sensörün ölçümleri ve aralık dışındaki bir değer halkayla, ısınma süresi de turuncu bantla işaretlenmiştir.

In [ ]:
if "truth" in paths:     # yalnızca sentetik veride bulunur
    truth = pd.read_csv(paths["truth"])["qc_truth"].to_numpy()
    pairs = {"warmup": "warmup", "spike_pm25": "spike_pm25_ugm3",
             "spike_pm10": "spike_pm10_ugm3", "spike_co2": "spike_co2_ppm",
             "flatline_co2": "flatline_co2", "gap": "gap"}
    rows = [dict(problem=k, planted=int((truth == k).sum()),
                 flagged=int(flags[v].sum()),
                 caught=int(((truth == k) & flags[v].to_numpy()).sum()))
            for k, v in pairs.items()]
    display(pd.DataFrame(rows))
v = df["visit"] == "B-R1-P1"
plt.figure(figsize=(9, 3))
plt.plot(df.loc[v, "seconds_since_arrival"] / 60, df.loc[v, "co2_ppm"],
         color="#0E7A5F")
bad = v & (flags["flatline_co2"] | flags["spike_co2_ppm"])
plt.scatter(df.loc[bad, "seconds_since_arrival"] / 60, df.loc[bad, "co2_ppm"],
            facecolors="none", edgecolors="#B3261E", s=50, label="işaretli")
plt.axvspan(0, 2, color="#eda100", alpha=0.15, label="ısınma")
plt.ylabel("CO₂ (ppm)"); plt.xlabel("varıştan sonraki dakika")
plt.legend(); plt.show()

### Alıştırma 1.A

Çözümlü örnekteki yedi ölçüm arasında 97,3 değerinin değiştirilmiş z-skorunu `robust_z` ile hesaplayın. Değeri `answer` değişkenine yazın.

In [ ]:
readings = np.array([14.2, 15.0, 13.8, 14.6, 97.3, 15.1, 14.4])
answer = None   # your answer
check("Alıştırma 1.A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
answer = robust_z(readings)[4]
print(round(answer, 2))
check("Alıştırma 1.A", answer)

### 1.3 YZ'den ikinci görüş: Isolation Forest

Kurallar, birinin önceden düşündüğü sorunları yakalar. Isolation Forest ise bütün kanallarda aynı anda geri kalanlardan kolayca ayrılan ölçümleri arar [2]. Veriyi rastgele değerlerden bölen rastgele ağaçlar kurar. Yalnızca birkaç bölmeden sonra tek başına kalan bir ölçüm olağan dışıdır. Eğitimden önce her kanal kendi ziyaretinin medyanından farkı olarak ifade edilir. Böylece orman kampüsleri ya da saatleri değil, aynı ziyaretteki ölçümleri karşılaştırır. `contamination=0.02` ayarı, ormandan ayrılması en kolay %2'lik ölçümü işaretlemesini ister. Hücre ormanın kaç ölçümü işaretlediğini, bunlardan kaçını bir kuralın da işaretlediğini ve yalnızca ormanın bulduğu birkaç ölçümü yazdırır. Ormanın işaretleri silinecek değil, saha notlarıyla gözden geçirilecek bir listedir.

In [ ]:
from sklearn.ensemble import IsolationForest

X_if = df[CHANNELS].copy()
X_if = (X_if - X_if.groupby(df["visit"]).transform("median")).fillna(0)
iso = IsolationForest(n_estimators=200, contamination=0.02,
                      random_state=RANDOM_STATE).fit(X_if)
df["if_flag"] = iso.predict(X_if) == -1
rule_any = flags.drop(columns=["high_rh"]).any(axis=1)
print("Isolation Forest işaretleri:", int(df["if_flag"].sum()))
print("bunlardan bir kuralın da işaretledikleri:",
      int((df["if_flag"] & rule_any).sum()))
display(df.loc[df["if_flag"] & ~rule_any,
               ["visit", "seconds_since_arrival", "pm25_ugm3", "co2_ppm"]].head(8))

**Eksik değer sıfır değildir.** Kampüs A'nın 2. turunda kayıt cihazı P5 noktasında yeniden başladı ve altı boş ölçüm bıraktı. Sonraki hücre boşluğun çevresindeki 22 ölçümü alır ve ortalama sıcaklığı iki kez hesaplar: Bir kez boş ölçümler atlanarak, bir kez de sıfır yazılarak. Aradaki fark, bir boşluğun veride neden boş kalması gerektiğini göstermektedir.

In [ ]:
gap = df[df["visit"] == "A-R2-P5"]["temperature_c"].iloc[30:52]
print("atlanınca:", round(gap.mean(), 2), "°C   sıfır yazılınca:",
      round(gap.fillna(0).mean(), 2), "°C")

### 1.4 Üç istasyon, tek hikâye

Saha turlarından önce üç istasyon iki saat boyunca yan yana çalıştırıldı. Aynı havayı ölçtükleri için aralarında kalıcı bir fark varsa bu, cihazın kendi sapmasıdır. s istasyonunun sapması, üç istasyonun ortalamasından ortalama farkıdır:

$$o_s = \frac{1}{T}\sum_{t=1}^{T}\bigl(x_s(t) - \bar{x}(t)\bigr)$$

Burada x<sub>s</sub>(t), s istasyonunun testin t. dakikasındaki ölçümü, x̄(t) aynı dakikada üç istasyonun ortalaması, T ise dakika sayısıdır. Sıcaklık, nem ve CO₂ için bu sapma istasyonun her saha ölçümünden çıkarılır. Partiküller için ise bir oran kullanılır, çünkü optik sensörlerin ölçümleri genellikle derişimle orantılı olarak farklılaşır. Bu durumda ölçümler, istasyonun üç istasyon ortalamasına göre ortalama oranına bölünür.

*Formüllerden koda: `colocation_offsets` sıcaklık, nem ve CO₂ için o değerini, partiküller için ise `pm_ratio` oranını hesaplar. `apply_offsets` ardından sapmaları çıkarır ve ölçümleri bu orana böler.*

In [ ]:
def colocation_offsets(coloc):
    """Her istasyonun üç istasyonun ortalamasından sapması."""
    wide = {ch: coloc.pivot(index="timestamp", columns="station", values=ch)
            for ch in ["temperature_c", "rel_humidity_pct", "pm25_ugm3", "co2_ppm"]}
    rows = []
    for s in sorted(coloc["station"].unique()):
        rows.append(dict(
            station=s,
            temperature_c=(wide["temperature_c"][s]
                           - wide["temperature_c"].mean(axis=1)).mean(),
            rel_humidity_pct=(wide["rel_humidity_pct"][s]
                              - wide["rel_humidity_pct"].mean(axis=1)).mean(),
            co2_ppm=(wide["co2_ppm"][s] - wide["co2_ppm"].mean(axis=1)).mean(),
            pm_ratio=(wide["pm25_ugm3"][s] / wide["pm25_ugm3"].mean(axis=1)).mean(),
        ))
    return pd.DataFrame(rows).set_index("station")

def apply_offsets(df, offsets):
    out = df.copy()
    st = out["station"]
    for ch in ["temperature_c", "rel_humidity_pct", "co2_ppm"]:
        out[ch] = out[ch] - st.map(offsets[ch])
    for ch in PM:
        out[ch] = out[ch] / st.map(offsets["pm_ratio"])
    return out


offsets = colocation_offsets(coloc)
display(offsets.round(3))

### 1.5 Nemli hava ve partikül sayacı

Nemli havada partiküller su alır ve büyür. Optik sensör bu büyümüş ıslak partikülleri görür ve kuru partiküllerin gerçek kütlesinden daha fazla kütle bildirir. Crilley vd. [3] tarafından kullanılan κ-Köhler düzeltmesi, ölçümü bir C büyüme çarpanına böler:

$$C(a_w) = 1 + \frac{\kappa\,\rho_w/\rho_p}{-1 + 1/a_w}, \qquad a_w = \mathrm{RH}/100$$

Burada a<sub>w</sub> suyun aktivitesidir ve bağıl nemin (RH) 100'e bölümüne eşittir. κ partiküllerin higroskopikliğini, ρ<sub>w</sub> = 1 g/cm³ suyun yoğunluğunu, ρ<sub>p</sub> ise kuru partiküllerin yoğunluğunu gösterir. Atölyede κ = 0,3 ve ρ<sub>p</sub> = 1,65 g/cm³ kullanılmaktadır. Örneğin %85 bağıl nemde C = 1 + (0,3 / 1,65) / (1/0,85 − 1) = 1 + 0,182 / 0,176 ≈ 2,03 olur. Yani düzeltilmiş ölçüm ham ölçümün yaklaşık yarısıdır. Hücre dört nem düzeyi için C değerini yazdırır. Gerçek bir çalışma ise κ değerini yerel hava için bir referans cihazla kestirmelidir.

In [ ]:
def humidity_factor(rh, kappa=0.3, rho_p=1.65):
    """Partikül kütlesinin nemle büyüme çarpanı C (Crilley vd., 2018)."""
    aw = np.clip(np.asarray(rh, dtype=float), 1, 99) / 100.0
    return 1.0 + (kappa / rho_p) / (-1.0 + 1.0 / aw)

def correct_pm(df):
    out = df.copy()
    c = humidity_factor(out["rel_humidity_pct"])
    for ch in PM:
        out[ch] = out[ch] / c
    return out


for rh in (50, 70, 85, 90):
    print(f"RH {rh} %  ->  C = {humidity_factor(rh):.2f}")

### 1.6 Temiz dakika tablosu

Bu hücre 1. kısmın adımlarını bir araya getirir. İşaretli değerler eksik yapılır, asla sıfır yapılmaz. Ardından istasyon sapmaları giderilir, partiküller neme göre düzeltilir ve her ziyaret, her kanalın medyanıyla dakika dakika özetlenir. Her ziyaretin ilk iki dakikası tablodan çıkar, çünkü ısınma işareti bu dakikaların bütün kanallarını boşaltır. Tablo `nokta_clean_minutes.csv` adıyla kaydedilir ve 2-4. kısımlar bu tablodan başlar. Sondaki küçük tablo, verideki en nemli sabah olan Kampüs C'nin 2. turunda ham ve düzeltilmiş PM2,5 değerlerini her noktanın bağıl nemiyle birlikte karşılaştırır.

In [ ]:
def minute_table(df):
    """Her ziyaretin her dakikası için bütün kanalların medyanı."""
    d = df.copy()
    d["minute"] = d["seconds_since_arrival"] // 60
    keys = ["campus", "round", "point_id", "point_type", "visit_order",
            "station", "visit", "minute"]
    m = d.groupby(keys)[CHANNELS].median().reset_index()
    t0 = d.groupby(keys)["timestamp"].min().reset_index(drop=True)
    m["hour"] = t0.dt.hour + t0.dt.minute / 60.0
    return m.dropna(subset=["temperature_c"]).reset_index(drop=True)


clean = clean_readings(df, flags)
adjusted_raw_pm = apply_offsets(clean, offsets)
adjusted = correct_pm(adjusted_raw_pm)
minutes = minute_table(adjusted)
minutes_raw_pm = minute_table(adjusted_raw_pm)
minutes.to_csv("nokta_clean_minutes.csv", index=False)
print(len(minutes), "temiz dakika nokta_clean_minutes.csv dosyasına kaydedildi")
c2 = (minutes.campus == "C") & (minutes["round"] == 2)
display(pd.DataFrame({
    "ham PM2,5": minutes_raw_pm[c2].groupby("point_type")["pm25_ugm3"].median(),
    "düzeltilmiş": minutes[c2].groupby("point_type")["pm25_ugm3"].median(),
    "RH %": minutes[c2].groupby("point_type")["rel_humidity_pct"].median(),
}).round(2))

**Gözlemle ve düşün.** Hangi işaretleri önce saha notlarıyla kontrol ederdiniz ve neden? Hangi sorunları bir kural asla bulamaz, ama noktadaki bir ekip üyesi hemen fark eder?

## 2. Analiz et: Adil karşılaştırma

*Atölye sitesi: [2. Bölüm](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/part2.html)*

Defterin 2. kısmı noktaları adil biçimde karşılaştırır. İki turlu tasarımla saatin etkisini giderir, tipik değerleri dayanıklı istatistiklerle anlatır, bir korelasyonu grupların içinde sınar ve çiy noktasıyla nemi sıcaklıktan ayırır.

### 2.1 Her noktanın arkasındaki saat

`visit_table` fonksiyonu her ziyareti temiz dakikalarının medyanıyla özetler ve ziyaretin saatini `hour` sütununda tutar. Grafik, Kampüs A'daki altı noktanın sıcaklığını saate göre gösterir. 1. tur düz, 2. tur kesikli çizgiyle çizilir ve her nokta kendi türünün rengini alır. Hava ısındığı için iki çizgi de sabah boyunca yükselme eğilimindedir, her nokta ise buna kendi etkisini ekler. Bir turda geç ziyaret edilen nokta diğer turda erken ziyaret edilir. Bu yüzden tek bir tur, yerin etkisini saatin etkisiyle karıştırır.

In [ ]:
def visit_table(minutes):
    keys = ["campus", "round", "point_id", "point_type", "visit_order"]
    v = minutes.groupby(keys)[CHANNELS + ["hour"]].median().reset_index()
    return v


visits = visit_table(minutes)
a = visits[visits.campus == "A"]
fig, ax = plt.subplots(figsize=(8, 3.6))
for rd, style in ((1, "-"), (2, "--")):
    r = a[a["round"] == rd].sort_values("hour")
    ax.plot(r["hour"], r["temperature_c"], style, color="gray", lw=1)
    colors = [TYPE_COLOR[t] for t in r.point_type]
    ax.scatter(r["hour"], r["temperature_c"], c=colors, s=70, zorder=3)
ax.set_xlabel("saat"); ax.set_ylabel("°C")
ax.set_title("Kampüs A: 1. tur (düz) ve 2. tur (kesikli)")
plt.show()

### 2.2 İki tur ortalaması ve referans noktası

$$\bar{x}_p = \frac{x_{p,1} + x_{p,2}}{2}, \qquad \Delta_p = \bar{x}_p - \bar{x}_{\text{ref}}$$

Burada x<sub>p,1</sub> ve x<sub>p,2</sub>, p noktasının 1. ve 2. turdaki ziyaret medyanları, x̄<sub>p</sub> bunların ortalaması, Δ<sub>p</sub> ise bu ortalamanın aynı kampüsteki sakin referans noktasının iki tur ortalamasından farkıdır. Sabahın saatte b °C ısındığını varsayalım. 1. turun k. sırasındaki ziyaret, 2. turun 5 − k. sırasındaki ziyaretle eşleşir. Böylece her nokta aynı toplam kaymayı alır ve kayma Δ<sub>p</sub> içinde birbirini götürür. `fair_table` fonksiyonu bu sütunları her kampüs ve nokta türü için verir. Çubuk grafik ardından üç kampüsün tamamında PM2,5 için Δ<sub>p</sub> değerini gösterir: Sıfırın sağındaki bir çubuk, referans noktasından daha fazla PM2,5 bulunan bir noktayı işaretler.

In [ ]:
def fair_table(visits, channel):
    """1. tur, 2. tur, ortalamaları ve referanstan fark."""
    w = visits.pivot_table(index=["campus", "point_type"], columns="round",
                           values=channel).rename(columns={1: "round1", 2: "round2"})
    w["two_round_mean"] = (w["round1"] + w["round2"]) / 2
    ref = w.xs("reference", level="point_type")["two_round_mean"]
    w["minus_reference"] = w["two_round_mean"] - ref.reindex(
        w.index.get_level_values("campus")).values
    return w.reset_index()


fair_t = fair_table(visits, "temperature_c")
display(fair_t[fair_t.campus == "A"].round(2))
fair_pm = fair_table(visits, "pm25_ugm3")
pivot = fair_pm.pivot(index="point_type", columns="campus", values="minus_reference")
pivot.loc[TYPE_ORDER].plot.barh(figsize=(7, 3.6),
                                color=["#2a78d6", "#eb6834", "#1baf7a"])
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("PM2,5 eksi referans noktası (µg/m³)")
plt.show()

### Alıştırma 2.A

`fair_t` tablosunu kullanarak Kampüs A'da yeşil alanın referans noktasından farkını bulun (°C).

In [ ]:
answer = None   # your answer
check("Alıştırma 2.A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
row = fair_t[(fair_t.campus == "A") & (fair_t.point_type == "green")]
answer = row["minus_reference"].iloc[0]
print(round(answer, 2))
check("Alıştırma 2.A", answer)

### 2.3 Tipik değerler

`add_anomalies` fonksiyonu her dakikayı kendi kampüsünün ve turunun medyanından farkı olarak ifade eder. Böylece yüksek kampüsteki serin bir sabah ile kıyıdaki ılık bir sabah aynı grafikte yer alabilir. Kutu grafikleri her nokta türü için bu PM2,5 farklarını gösterir. Kutu, birinci çeyrekten üçüncü çeyreğe kadar dakikaların ortadaki yarısını kapsar. Kutunun içindeki çizgi medyanı, küçük üçgen ise ortalamayı gösterir. Kalabalık girişin ve yolun yakınında kısa süreli bulutlar üst bıyığı uzatır ve ortalamayı medyanın üstüne çeker. Grafiğin altındaki tablo her nokta türü için iki değeri de verir.

In [ ]:
def add_anomalies(minutes):
    m = minutes.copy()
    grp = m.groupby(["campus", "round"])
    m["temp_anomaly"] = m["temperature_c"] - grp["temperature_c"].transform("median")
    m["pm25_anomaly"] = m["pm25_ugm3"] - grp["pm25_ugm3"].transform("median")
    m["co2_anomaly"] = m["co2_ppm"] - grp["co2_ppm"].transform("median")
    m["log10_lux"] = np.log10(m["light_lux"].fillna(LUX_MAX))
    return m


anom = add_anomalies(minutes)
fig, ax = plt.subplots(figsize=(9, 3.6))
data = [anom.loc[anom.point_type == t, "pm25_anomaly"].dropna() for t in TYPE_ORDER]
bp = ax.boxplot(data, patch_artist=True, showmeans=True)
for patch, t in zip(bp["boxes"], TYPE_ORDER):
    patch.set_facecolor(TYPE_COLOR[t])
ax.set_xticks(range(1, 7), [TYPE_NAME[t] for t in TYPE_ORDER])
ax.set_ylabel("PM2,5 eksi kampüs-tur medyanı (µg/m³)")
plt.show()
display(anom.groupby("point_type")["pm25_anomaly"].agg(["median", "mean"]).round(2))

### 2.4 İlişki mi, neden mi?

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\,\sqrt{\sum_i (y_i - \bar{y})^2}}$$

Burada x<sub>i</sub>, i. dakikanın sıcaklığı, y<sub>i</sub> ise hava basıncıdır. x̄ ve ȳ bunların ortalamalarıdır. Pearson korelasyonu r, −1 ile +1 arasında değişir ve noktaların oluşturduğu buluta bir doğrunun ne kadar iyi uyduğunu ölçer. Hücre r değerini önce bütün temiz dakikalar için, ardından her kampüs ve turun kendi içinde hesaplar. Bütün dakikalarda r belirgin biçimde pozitiftir, ancak her grubun içinde negatiftir. Kıyı kampüsü hem daha sıcak hem de daha alçaktadır. Bu yüzden dakikaları yüksek sıcaklığı yüksek basınçla birleştirir ve pozitif değeri tek başına bu durum yaratır. Bir grubun içinde ise hava ısınırken basınç sabah boyunca yavaşça düşer. Bu tersine dönüş Simpson paradoksu olarak bilinmektedir. Sondaki ısı haritası on kanalın tamamı arasındaki korelasyonları gösterir.

In [ ]:
def pearson_r(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    return float(np.corrcoef(x[ok], y[ok])[0, 1])


r_all = pearson_r(minutes.temperature_c, minutes.pressure_hpa)
print("bütün dakikalar:", round(r_all, 3))
for (c, rd), g in minutes.groupby(["campus", "round"]):
    print(f"{c}{rd}:", round(pearson_r(g.temperature_c, g.pressure_hpa), 3))
corr = minutes[CHANNELS].corr()
plt.figure(figsize=(6.5, 5.2))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(10), CHANNELS, rotation=60, ha="right")
plt.yticks(range(10), CHANNELS)
plt.colorbar(label="r"); plt.title("On kanalın korelasyonu")
plt.show()

### 2.5 Çiy noktası

Bağıl nem sıcaklığa bağlıdır: Aynı hava soğuduğunda daha yüksek bir bağıl nem gösterir. Çiy noktası bu bağımlılığı ortadan kaldırır. Çiy noktası, havadaki su buharının yoğuşması için havanın soğuması gereken sıcaklıktır. Bu yüzden yalnızca havadaki su miktarı değiştiğinde değişir. Magnus biçimi [4], çiy noktasını sıcaklıktan ve bağıl nemden hesaplar:

$$\gamma = \ln\frac{\mathrm{RH}}{100} + \frac{a\,T}{b + T}, \qquad T_d = \frac{b\,\gamma}{a - \gamma}, \quad a = 17{,}625,\ b = 243{,}04\ ^\circ\mathrm{C}$$

Burada T °C cinsinden hava sıcaklığı, RH yüzde olarak bağıl nem, γ bir ara büyüklük, T<sub>d</sub> ise °C cinsinden çiy noktasıdır. a ve b sabitleri Alduchov ve Eskridge'in [4] uyarladığı değerlerdir. Hücre her temiz dakikaya çiy noktasını ekler ve her kampüs, tur ve nokta türü için medyanını yazdırır.

In [ ]:
def dew_point(t, rh):
    """Magnus biçiminden çiy noktası, °C (Alduchov ve Eskridge, 1996)."""
    a, b = 17.625, 243.04
    g = np.log(np.asarray(rh, dtype=float) / 100.0) + a * t / (b + t)
    return b * g / (a - g)


minutes["dew_point_c"] = dew_point(minutes.temperature_c, minutes.rel_humidity_pct)
display(minutes.pivot_table(index=["campus", "round"], columns="point_type",
                            values="dew_point_c", aggfunc="median").round(1))

### Alıştırma 2.B

20 °C sıcaklık ve %50 bağıl nemdeki havanın çiy noktası nedir? `dew_point` fonksiyonunu kullanın.

In [ ]:
answer = None   # your answer
check("Alıştırma 2.B", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
answer = dew_point(20.0, 50.0)
print(round(answer, 2))
check("Alıştırma 2.B", answer)

**Gözlemle ve düşün.** Çiy noktası hangi kampüs ve turda en yüksek ve bu, o sabahın havası hakkında ne söyler? Yeşil alan neden biraz daha yüksektir?

## 3. Keşfet: YZ örüntü bulur

*Atölye sitesi: [3. Bölüm](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/part3.html)*

Defterin 3. kısmı, k-ortalamalar algoritmasının temiz dakikaları nerede ölçüldüklerini bilmeden gruplamasını sağlar [5]. Ardından grup sayısını iki skorla seçer ve altı kanalı temel bileşenler analiziyle sıkıştırır [7].

### 3.1 Standartlaştır

Her dakika üç özellikle tanımlanır: Kampüsünün ve turunun medyanından sıcaklık farkı olan `temp_anomaly`, ışık şiddetinin logaritması olan `log10_lux` ve PM2,5 farkı olan `pm25_anomaly`. Işık için logaritma kullanılır, çünkü aydınlık düzeyi gölgede on bin lüksün altından tam güneşte sensörün 65.535 lükslük sınırına kadar uzanır. k-ortalamalar uzaklıkları ölçmeden önce her özellik standartlaştırılır:

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

Burada x<sub>ij</sub>, i. dakikada j. özelliğin değeri, μ<sub>j</sub> ve σ<sub>j</sub> ise o özelliğin ortalaması ve standart sapmasıdır. Bu adımdan sonra her özelliğin ortalaması 0, standart sapması 1 olur. Böylece bir derece sıcaklık ile bir mikrogram partikül uzaklıklarda artık farklı ağırlık taşımaz. Hücre üç özelliğin μ ve σ değerlerini yazdırır.

In [ ]:
def standardize(X):
    mu, sd = X.mean(axis=0), X.std(axis=0, ddof=0)
    return (X - mu) / sd, mu, sd


CLUSTER_FEATURES = ['temp_anomaly', 'log10_lux', 'pm25_anomaly']
X = anom[CLUSTER_FEATURES].to_numpy()
Z, mu, sd = standardize(X)
print("μ =", mu.round(3), "  σ =", sd.round(3))

### 3.2 k = 4 ile k-ortalamalar

k-ortalamalar, standartlaştırılmış uzaya k merkez yerleştirir ve hiçbir şey değişmeyene kadar iki adımı tekrarlar [5]. Önce her dakika en yakın merkeze atanır. Ardından her merkez kendisine atanan dakikaların ortalamasına taşınır. Hücre scikit-learn kütüphanesindeki `KMeans` sınıfını dört grup ve on rastgele başlangıçla çalıştırır ve en iyi sonucu tutar. Grupları en serin merkezden en sıcak merkeze doğru numaralandırır ve merkezleri sensör birimlerine geri çevirir. Çapraz tablo ardından her grubu hangi nokta türlerinin doldurduğunu sayar. 3B grafik, atölye sitesindekiyle aynı bulutu gösterir ve fareyle döndürülebilir.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(Z)
order = np.argsort(km.cluster_centers_[:, 0])          # en serin grup önce
rename = {old: new + 1 for new, old in enumerate(order)}
anom["group"] = pd.Series(km.labels_).map(rename).to_numpy()
centers = pd.DataFrame(km.cluster_centers_[order] * sd + mu,
                       columns=CLUSTER_FEATURES, index=[1, 2, 3, 4]).round(2)
display(centers)
display(pd.crosstab(anom["group"], anom["point_type"])[TYPE_ORDER])

In [ ]:
if HAS_PLOTLY:
    fig = px.scatter_3d(anom, x="temp_anomaly", y="log10_lux", z="pm25_anomaly",
                        color=anom["group"].astype(str), opacity=0.75,
                        hover_data=["campus", "round", "point_type"],
                        title="Temiz dakikaların k-ortalamalar grupları")
    fig.update_traces(marker_size=3)
    fig.show()
else:
    ax = plt.figure(figsize=(6, 5)).add_subplot(projection="3d")
    ax.scatter(anom.temp_anomaly, anom.log10_lux, anom.pm25_anomaly,
               c=anom.group, s=8)
    ax.set_xlabel("ΔT"); ax.set_ylabel("log10 lux"); ax.set_zlabel("ΔPM2.5")
    plt.show()

### 3.3 Kaç grup?

k-ortalamalar grup sayısının önceden verilmesini ister. Hücre algoritmayı 2'den 6'ya kadar her k için çalıştırır ve iki skor kaydeder. Dakikaların merkezlerine toplam kare uzaklığı olan J, k büyüdükçe her zaman düşer, çünkü merkez sayısı arttıkça merkezler dakikalara yaklaşır. Bu yüzden k değerini tek başına J seçemez. Siluet ise her dakika için kendi grubuna ortalama uzaklığı, en yakın diğer gruba ortalama uzaklıkla −1 ile +1 arasındaki bir ölçekte karşılaştırır [6]. Daha yüksek bir siluet değeri, birbirinden daha iyi ayrılmış gruplar anlamına gelir. Birkaç k değeri neredeyse aynı skoru aldığında seçim, analistin gerekçelendirmesi gereken bir karardır. Bu defter, dört tür mikroiklim olarak okunabilen dört grupla çalışmaktadır.

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    rows.append(dict(k=k, J=round(m.inertia_, 1),
                     silhouette=round(silhouette_score(Z, m.labels_), 3)))
scores = pd.DataFrame(rows)
display(scores)

### Alıştırma 3.A

`scores` tablosunda hangi k değeri en yüksek siluet skoruna sahip?

In [ ]:
answer = None   # your answer
check("Alıştırma 3.A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
answer = int(scores.loc[scores.silhouette.idxmax(), "k"])
print(answer)
check("Alıştırma 3.A", answer)

### 3.4 Temel bileşenler analizi

Temel bileşenler analizi (PCA), standartlaştırılmış birkaç kanalı, koruduğu değişim miktarına göre sıralanan yeni eksenlere, yani temel bileşenlere dönüştürür [7]. Her bileşen kanalların ağırlıklı bir toplamıdır ve bu ağırlıklara yük denir. Hücre altı kanalı standartlaştırır: Sıcaklık, PM2,5 ve CO₂ farkları, bağıl nem, ışık şiddetinin logaritması ve UVA ışınımı. CO₂ değeri olmayan bir dakika kendi kampüsünün ve turunun medyanına yerleştirilir, yani CO₂ farkı sıfır kabul edilir. Hücre ilk üç bileşenin her birinin koruduğu değişim payını ve yük tablosunu yazdırır. Üç bileşen birlikte değişimin yaklaşık %86'sını korur. Pozitif ya da negatif büyük bir yük, bir bileşenin esas olarak hangi kanalları anlattığını gösterir. 3B grafik son olarak dakikaları bu üç yeni eksende nokta türüne göre renklendirerek çizer.

In [ ]:
from sklearn.decomposition import PCA

feats = ["temp_anomaly", "rel_humidity_pct", "pm25_anomaly", "co2_anomaly",
         "log10_lux", "uva_wm2"]
P = anom[feats].copy()
# eksik dakika medyana yerleştirilir
P["co2_anomaly"] = P["co2_anomaly"].fillna(0.0)
Zp, _, _ = standardize(P.to_numpy())
pca = PCA(n_components=3, random_state=RANDOM_STATE).fit(Zp)
print("değişimin payı:", pca.explained_variance_ratio_.round(3))
loadings = pd.DataFrame(pca.components_, columns=feats, index=["PC1", "PC2", "PC3"])
display(loadings.round(2))
S = pca.transform(Zp)
if HAS_PLOTLY:
    fig = px.scatter_3d(x=S[:, 0], y=S[:, 1], z=S[:, 2], color=anom["point_type"],
                        color_discrete_map=TYPE_COLOR,
                        labels={"x": "PC1", "y": "PC2", "z": "PC3"})
    fig.update_traces(marker_size=3)
    fig.show()

**Gözlemle ve düşün.** Hangi grup birkaç nokta türünü karıştırıyor ve bu dakikaların ortak yanı ne? Aynı dört gruba bulutlu bir günde de güvenir miydiniz?

## 4. Açıkla: Burada neden yüksek?

*Atölye sitesi: [4. Bölüm](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/part4.html)*

Defterin 4. kısmı, her temiz dakikanın PM2,5 değerini tahmin eden bir rastgele orman eğitir [8]. Modeli üç farklı biçimde test eder, hangi girdilere dayandığını sorar ve cevabını kaydırıcılar, kısmi bağımlılık ve bir 3B yüzeyle gösterir.

### 4.1 Model tablosu

`model_table` fonksiyonu her temiz dakikayı kendi ziyaretinin saha formuyla birleştirir ve modelin sekiz girdisini oluşturur. 15 dakikada sayılan araç ve kişi sayıları dakika başına orana çevrilir. Güneş durumu bir sayıya dönüştürülür: Gölge için 0, kısmi güneş için 0,5 ve tam güneş için 1. Bir sabahın arka plan PM2,5 değeri, o kampüs ve turda referans noktasında ölçülen PM2,5 değerlerinin medyanıdır. Sıcaklık, bağıl nem ve saat dakikanın kendisinden gelir. Hedef değişken, dakikanın düzeltilmiş PM2,5 değeridir. Hücre tablodaki dakika ve ölçüm noktası sayısını yazdırır.

In [ ]:
def model_table(minutes, forms, target_minutes=None):
    m = minutes.merge(
        forms[["campus", "round", "point_id", "vehicles_15min", "people_15min",
               "green_cover_pct", "sun_condition"]],
        on=["campus", "round", "point_id"], how="left")
    m["vehicles_per_min"] = m["vehicles_15min"] / 15.0
    m["people_per_min"] = m["people_15min"] / 15.0
    m["sun_index"] = m["sun_condition"].map({"shade": 0.0, "partial": 0.5,
                                             "sun": 1.0})
    ref = (m[m["point_type"] == "reference"]
           .groupby(["campus", "round"])["pm25_ugm3"].median()
           .rename("background_pm25"))
    m = m.merge(ref, on=["campus", "round"], how="left")
    return m


MODEL_FEATURES = ['vehicles_per_min', 'people_per_min', 'green_cover_pct',
    'sun_index', 'rel_humidity_pct', 'temperature_c', 'hour',
    'background_pm25']
table = model_table(minutes, forms).dropna(subset=MODEL_FEATURES + ["pm25_ugm3"])
Xm = table[MODEL_FEATURES].to_numpy()
y = table["pm25_ugm3"].to_numpy()
groups = (table.campus + table.point_id).to_numpy()
print(len(y), "dakika,", len(set(groups)), "ölçüm noktası")
display(table[MODEL_FEATURES + ["pm25_ugm3"]].head(3))

### 4.2 Üç skor

Belirlilik katsayısı R², modelin hatalarını verinin yayılımıyla karşılaştırır:

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

Burada y<sub>i</sub>, i. dakikada ölçülen PM2,5 değeri, ŷ<sub>i</sub> modelin tahmini, ȳ ise bütün ölçümlerin ortalamasıdır. R² = 1 kusursuz tahmin anlamına gelir. R² = 0 ise modelin her seferinde ortalamayı tahmin etmekten daha iyi olmadığını gösterir. Orman, her biri en çok altı düzey derinliğinde 40 ağaçtan oluşmaktadır. Hücre üç skor hesaplar. İlki eğitim dakikalarının kendisi üzerinde ölçülür ve her zaman fazla iyimserdir. İkincisi rastgele dakikalar üzerinde beş katlı çapraz doğrulamadan gelir. Bu durumda aynı ziyaretin dakikaları hem eğitim hem de test kısmında bulunabilir. Üçüncüsü her seferinde bir ölçüm noktasını dışarıda bırakır, böylece model test edildiği noktayı hiç görmez. Aynı noktadan gelen çok sayıda dakika birbirinden bağımsız gözlemler değildir. Bu yüzden modelin yeni bir yerde nasıl çalışacağını yalnızca üçüncü skor anlatır [9].

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict


def make_forest():
    return RandomForestRegressor(n_estimators=40, max_depth=6, min_samples_leaf=5,
                                 random_state=RANDOM_STATE, n_jobs=-1)


forest = make_forest().fit(Xm, y)
p_random = cross_val_predict(make_forest(), Xm, y,
                             cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
p_group = cross_val_predict(make_forest(), Xm, y, cv=LeaveOneGroupOut(),
                            groups=groups)
scores_cv = pd.Series({"kendi eğitim dakikaları": r2_score(y, forest.predict(Xm)),
                       "rastgele dakikalar (5 katlı)": r2_score(y, p_random),
                       "bir noktayı dışarıda bırak": r2_score(y, p_group)}).round(3)
display(scores_cv.to_frame("R²"))

### 4.3 Permütasyon önemi

Permütasyon önemi, bir girdi işe yaramaz hâle getirildiğinde modelin ne kadar kötüleştiğini sorar. O girdinin değerleri dakikalar arasında karıştırılır, böylece girdinin PM2,5 ile bağı koparılır ve R² değerindeki düşüş ölçülür. Testin dürüst kalması için orman iki kampüste eğitilir ve önem değerleri üçüncü kampüste ölçülür. Bu işlem her kampüs için tekrarlanır ve üç sonucun ortalaması alınır. Çubuk grafik her girdi için R² değerindeki ortalama düşüşü gösterir. Büyük bir düşüş, modelin o girdiye dayandığını gösterir. Ancak bu, o girdinin gerçek dünyada PM2,5'e neden olduğunu kanıtlamaz.

In [ ]:
from sklearn.inspection import permutation_importance


def campus_importance(tab, target="pm25_ugm3"):
    X_, y_ = tab[MODEL_FEATURES].to_numpy(), tab[target].to_numpy()
    res = []
    for c in sorted(tab.campus.unique()):
        train = (tab.campus != c).to_numpy()
        m = make_forest().fit(X_[train], y_[train])
        pi = permutation_importance(m, X_[~train], y_[~train], n_repeats=10,
                                    random_state=RANDOM_STATE)
        res.append(pi.importances_mean)
    return pd.Series(np.mean(res, axis=0), index=MODEL_FEATURES).sort_values()


imp = campus_importance(table)
imp.plot.barh(figsize=(6, 3.4), color="#A65400")
plt.xlabel("karıştırıldığında R² düşüşü"); plt.show()

### Alıştırma 4.A

Hangi girdinin önemi en büyük? Adını metin olarak yazın, örneğin `"hour"`. Örnek çözüm, önemi nem düzeltmesi yapılmamış ham PM2,5 için (`minutes_raw_pm`) de hesaplar. Böylece iki sütun karşılaştırılabilir.

In [ ]:
answer = None   # your answer
check("Alıştırma 4.A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
answer = imp.idxmax()
print(answer)
check("Alıştırma 4.A", answer)
raw_table = model_table(minutes_raw_pm, forms)
raw_table = raw_table.dropna(subset=MODEL_FEATURES + ["pm25_ugm3"])
both = pd.DataFrame({"düzeltilmiş": imp, "ham": campus_importance(raw_table)})
display(both.round(3))

### 4.4 Ya şöyle olsaydı?

Kaydırıcılar eğitilmiş ormanın girdilerini değiştirir, grafiğin başlığı da modelin tahminini verir. Her kaydırıcı kendi girdisinin medyanından başlar ve ölçülen verinin aralığında kalır, çünkü bir orman gördüğünün ötesini tahmin edemez. Her çubuk, yalnızca o girdi medyanına dönseydi ve diğerleri kaydırıcı değerlerinde kalsaydı tahminin ne kadar değişeceğini gösterir. Sağa uzanan çubuklar tahmini yukarı iten, sola uzananlar ise aşağı çeken girdileri gösterir.

In [ ]:
import ipywidgets as widgets

med = np.median(Xm, axis=0)
sliders = {f: widgets.FloatSlider(value=float(med[i]), min=float(Xm[:, i].min()),
                                  max=float(Xm[:, i].max()),
                                  step=float((Xm[:, i].max() - Xm[:, i].min()) / 50),
                                  description=f[:18], continuous_update=False)
           for i, f in enumerate(MODEL_FEATURES)}


def what_if(**values):
    x = np.array([values[f] for f in MODEL_FEATURES])
    p = forest.predict([x])[0]
    effect = {}
    for i, f in enumerate(MODEL_FEATURES):
        x2 = x.copy(); x2[i] = med[i]
        effect[f] = p - forest.predict([x2])[0]
    s = pd.Series(effect).sort_values()
    colors = ["#1F5F8B" if v < 0 else "#A65400" for v in s]
    s.plot.barh(figsize=(6, 3.2), color=colors)
    plt.title(f"tahmin edilen PM2,5: {p:.1f} µg/m³"); plt.show()


out = widgets.interactive_output(what_if, sliders)
display(widgets.VBox(list(sliders.values())), out)

### 4.5 Kısmi bağımlılık

Kısmi bağımlılık, bir girdi değişirken modelin ortalama cevabını gösterir [10]:

$$\hat{f}_S(x_S) = \frac{1}{n}\sum_{i=1}^{n} \hat{f}\bigl(x_S, \mathbf{x}_C^{(i)}\bigr)$$

Burada x<sub>S</sub> incelenen girdi, **x**<sub>C</sub><sup>(i)</sup> i. dakikanın diğer girdileri, n ise dakika sayısıdır. x<sub>S</sub> girdisinin her değeri için model, bütün dakikaları x<sub>S</sub> bu değere eşitlenmiş olarak tahmin eder ve tahminlerin ortalamasını alır. Hücre dakikadaki araç sayısı, arka plan PM2,5 değeri, dakikadaki kişi sayısı ve yeşil örtü için eğrileri çizer. `kind="both"` ayarıyla ince çizgiler örneklenen 60 dakikayı, kalın çizgi ise bunların ortalamasını gösterir. Böylece grafik tek tek dakikaların ortalama cevaptan ne kadar ayrıldığını da gösterir.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(12, 3.2))
PartialDependenceDisplay.from_estimator(
    forest, Xm, features=[0, 7, 1, 2], feature_names=MODEL_FEATURES, ax=ax,
    kind="both", subsample=60, random_state=RANDOM_STATE)
plt.tight_layout(); plt.show()

### 4.6 Model yüzeyi 3B

Son hücre ormanı, dakikadaki araç sayısı ile arka plan PM2,5 değerinin 30 × 30'luk bir ızgarasında değerlendirir. Bu sırada diğer altı girdi medyanlarında kalır. Sonuç, fareyle döndürülebilen bir yüzey olarak çizilir. Yüzey düzgün bir biçimde değil, basamaklar hâlinde yükselir, çünkü her ağaç girdilerini belirli eşik değerlerinden böler.

In [ ]:
ix = MODEL_FEATURES.index("vehicles_per_min")
iy = MODEL_FEATURES.index("background_pm25")
gx = np.linspace(Xm[:, ix].min(), Xm[:, ix].max(), 30)
gy = np.linspace(Xm[:, iy].min(), Xm[:, iy].max(), 30)
grid = np.tile(med, (900, 1))
grid[:, ix] = np.repeat(gx, 30)
grid[:, iy] = np.tile(gy, 30)
Zs = forest.predict(grid).reshape(30, 30).T
if HAS_PLOTLY:
    fig = go.Figure(go.Surface(x=gx, y=gy, z=Zs, colorscale="Blues"))
    fig.update_layout(scene=dict(xaxis_title="vehicles_per_min",
                                 yaxis_title="background_pm25", zaxis_title="PM2.5"),
                      height=520,
                      title="Tahmin edilen PM2,5, diğer girdiler medyanda")
    fig.show()
else:
    ax = plt.figure(figsize=(7, 5)).add_subplot(projection="3d")
    GX, GY = np.meshgrid(gx, gy)
    ax.plot_surface(GX, GY, Zs, cmap="Blues"); plt.show()

**Gözlemle ve düşün.** Orman, sentetik veriyi üreten kuralları buldu. Aynı öneme gerçek bir kampüste inanmak için hangi saha gözlemine ihtiyaç duyardınız?

## 5. Paylaş ve aksiyona dönüştür

*Atölye sitesi: [5. Bölüm](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/part5.html)*

Defterin 5. kısmı sonuçları bir rapora hazırlar. Analizden bir olgu tablosu oluşturur, bu olguları bir YZ asistanına veren bir istem yazar, YZ ile yazılmış bir metindeki her sayıyı tabloyla denetler ve 28 Kasım çalıştayı için bir çözüm kartı yazdırır.

### 5.1 Olgu tablosu

Olgu tablosu, bir raporun kullanabileceği sayıları bir araya getirir. `CAMPUS` değişkeninde belirtilen kampüs için her nokta türünde PM2,5, sıcaklık ve CO₂ değerlerini verir: İki turun ziyaret medyanları, bunların ortalaması ve referans noktasından fark. Tablo bir ondalık basamağa yuvarlanır ve bir CSV dosyası olarak kaydedilir. Böylece bir rapora eklenebilir ya da bir asistana yapıştırılabilir.

In [ ]:
CAMPUS = "B"
facts = pd.concat([fair_table(visits, ch).assign(channel=ch)
                   for ch in ["pm25_ugm3", "temperature_c", "co2_ppm"]])
facts = facts[facts.campus == CAMPUS][["channel", "point_type", "round1", "round2",
                                       "two_round_mean", "minus_reference"]].round(1)
facts.to_csv(f"nokta_facts_{CAMPUS}.csv", index=False)
display(facts)

### 5.2 Güvenli bir istem

Güvenli bir istem, asistandan sayıları bilmesini istemez. Sayıları ona verir, görevi tanımlar ve kuralları yazılı olarak belirtir. `make_prompt` fonksiyonu bir rolü, bir görevi, olgu tablosunu ve `RULES` listesindeki yedi kuralı birleştirir. Kurallar kampüslerin sıralanmasını ve değerlerin 24 saatlik ya da yıllık kılavuzlarla karşılaştırılmasını yasaklar. Ayrıca her değerin ortalama alma süresinin belirtilmesini, gözlemin, yorumun ve önerinin birbirinden ayrılmasını ve nedenler tartışılırken temkinli ifadeler kullanılmasını ister. Yazdırılan istem herhangi bir asistana kopyalanabilir, bir ekip de kuralları kendi raporu için genişletebilir.

In [ ]:
RULES = [
    "Yalnızca tablodaki sayıları kullan, eksik bir değer için 'ölçülmedi' yaz.",
    "Kampüsleri, üniversiteleri ya da şehirleri sıralama.",
    "Her değerin 15 dakikalık bir ziyaretin medyanı olduğunu belirt.",
    "Değerleri 24 saatlik ya da yıllık kılavuzlarla karşılaştırma.",
    "Gözlemi, yorumu ve öneriyi ayrı paragraflarda tut.",
    "Düşük maliyetli sensörleri, nem düzeltmesini ve sapmaları belirt.",
    "Nedenler için temkinli ifadeler kullan."]


def table_text(frame):
    try:
        return frame.to_markdown(index=False)          # tabulate paketi gerekir
    except ImportError:
        return frame.to_string(index=False)


def make_prompt(facts, campus, audience="bir kampüs çevre raporu"):
    lines = ["Rol: Kampüs ölçümlerini raporlayan bir öğrenci ekibine yardım et.",
             f"Görev: Kampüs {campus} verisiyle {audience} yaz.",
             "Veri:", table_text(facts), "Kurallar:"]
    return "\n".join(lines + ["- " + r for r in RULES])


print(make_prompt(facts, CAMPUS))

### 5.3 Sayı denetleyicisi

Denetleyici bir metni okur, içindeki her sayıyı listeler ve olgu tablosunda bulunmayan sayıları işaretler. Bir sayı, tablodaki bir değere 0,05'ten daha yakınsa desteklenmiş sayılır. Bu pay, yuvarlamadan doğan farkları karşılar. `text` değişkenindeki örnek metin, atölye sitesinin 5. Bölümündeki bazı hataları tekrarlar. Bunun yerine YZ ile yazılmış kendi metninizi de yapıştırabilirsiniz. İşaretli bir sayı her zaman yanlış değildir, örneğin bir kılavuz değeri ya da bir yıl olabilir. Ancak işaretli her sayının bir kaynağı olmalıdır.

In [ ]:
def audit_numbers(text, facts, tol=0.051):
    """Metindeki sayıları ve tablonun onları destekleyip desteklemediğini verir."""
    allowed = []
    for v in np.ravel(facts.select_dtypes("number").to_numpy()):
        if np.isfinite(v):
            allowed.append(float(v))
    rows = []
    for m in NUMBER.finditer(text):
        val = float(m.group().replace(",", "."))
        hit = any(abs(val - a) <= tol for a in allowed)
        rows.append(dict(number=m.group(), supported=hit,
                         context=text[max(0, m.start() - 30):m.end() + 30]))
    return pd.DataFrame(rows)


NUMBER = re.compile(r"(?<![\w.,])-?\d+(?:[.,]\d+)?")
text = ("Kampüs B'de en yüksek PM2,5, iki tur ortalaması 21,7 µg/m³ ile "
        'yoldaydı ve referansın 7,8 µg/m³ üstündeydi. Yeşil alan 6,1 µg/m³ '
        "ile en temiz noktaydı. Yolun yakınında PM2,5, DSÖ'nün 15 µg/m³ olan "
        '24 saatlik kılavuz değerini aştı. 2024 tarihli bir rapor bunu '
        'doğruluyor. Güneşli meydan 3,6 °C daha sıcaktı.')
audit = audit_numbers(text, facts)
display(audit)
print("tabloda desteği olmayan sayılar:",
      int((~audit.supported).sum()))

### Alıştırma 5.A

Denetleyici `text` içindeki kaç sayıyı desteksiz olarak işaretliyor?

In [ ]:
answer = None   # your answer
check("Alıştırma 5.A", answer)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; ancak denedikten sonra açın)
answer = int((~audit.supported).sum())
print(answer)
check("Alıştırma 5.A", answer)

### 5.4 Çözüm kartı

28 Kasım'da NOKTA Ulusal Çevre Çalıştayı üç tematik masada çalışır ve oraya getirilen her öneri beş soruyu cevaplar: Sorun, dayanak, çözüm, uygulanabilirlik ve gençlerin rolü. `solution_card` fonksiyonu bu beş cevabı bir kart olarak yazar. Örnek, olgu tablosundaki güneşli beton meydandan başlar ve dayanağını, yani referans noktasından farkı, doğrudan analizden alır. Bir ekip diğer dört cevabı kendi cümleleriyle değiştirir.

In [ ]:
def solution_card(table, problem, evidence, solution, feasibility, youth):
    parts = [("Sorun", problem), ("Dayanak", evidence), ("Çözüm", solution),
             ("Uygulanabilirlik", feasibility), ("Gençlerin rolü", youth)]
    return f"### {table}\n\n" + "\n\n".join(f"**{k}.** {v}" for k, v in parts)


row = facts[(facts.channel == "temperature_c") & (facts.point_type == "sunny_hard")]
card = solution_card(
    "Mikroiklim ve yeşil kampüs",
    "Beton meydan güneşte ısınıyor.",
    f"İki tur ortalaması {row.minus_reference.iloc[0]:+.1f} °C"
    " referans noktasına göre.",
    "Bankların üzerine gölgelik ve bir sıra ağaç.",
    "Gölgelikler düşük maliyetlidir, ağaçlar ise bir mevsim gerektirir.",
    "Öğrenciler bir yıl sonra aynı protokolle yeniden ölçer.")
display(Markdown(card))

## Uygulama görevleri

Aşağıdaki her görev, analizi bir alanın bakış açısından sürdürür. Görev bu defterdeki tablolardan başlar ve bunlara küçük bir saha çalışması ya da bir hesap ekler.

| Alan | Görev |
|---|---|
| Mühendislik | Güneşli meydandaki gerçek sıcaklığı sensör gövdesine vuran güneşten ayıran bir test tasarlayın. |
| Sağlık bilimleri | Bir öğrencinin her gün otobüs durağında ne kadar beklediğini ve PM2,5 farkının bu süre için ne anlama geldiğini kestirin. |
| Sosyal bilimler | On öğrenciye kampüste havayı en kötü nerede hissettiklerini sorun ve cevaplarını adil tabloyla karşılaştırın. |
| İletişim | Bir sonucu ortalama alma süresini ve belirsizliğini gösteren bir kongre şekline dönüştürün. |
| İktisat ve işletme | Güneşli meydan için gölgelik ve ağaçların on yıllık derece başına maliyetini karşılaştırın. |
| Eğitim | Öğrencilerin iki turlu tasarımı bir termometreyle tekrarladığı 40 dakikalık bir ders planlayın. |

## Defterden sonra

Oturumun saha raporu için [NOKTA Laboratuvarının değerlendirmesini](https://utkukose.github.io/env-data-ai-nokta2026-NB-workshop/tr/lab.html#debrief) açın. 21-24 Kasım analiz günleri için 0. kısımda `DATA_SOURCE = "own"` ayarını yapın ve defteri kendi NOKTA dosyalarınızla çalıştırın.

**Kaynaklar**

1. Leys, C., Ley, C., Klein, O., Bernard, P., & Licata, L. (2013). Detecting outliers: Do not use standard deviation around the mean, use absolute deviation around the median. *Journal of Experimental Social Psychology, 49*(4), 764–766. https://doi.org/10.1016/j.jesp.2013.03.013
2. Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation forest. In *Proceedings of the 8th IEEE International Conference on Data Mining (ICDM 2008)* (pp. 413–422). IEEE. https://doi.org/10.1109/ICDM.2008.17
3. Crilley, L. R., Shaw, M., Pound, R., Kramer, L. J., Price, R., Young, S., Lewis, A. C., & Pope, F. D. (2018). Evaluation of a low-cost optical particle counter (Alphasense OPC-N2) for ambient air monitoring. *Atmospheric Measurement Techniques, 11*(2), 709–720. https://doi.org/10.5194/amt-11-709-2018
4. Alduchov, O. A., & Eskridge, R. E. (1996). Improved Magnus form approximation of saturation vapor pressure. *Journal of Applied Meteorology, 35*(4), 601–609. https://doi.org/10.1175/1520-0450(1996)035<0601:IMFAOS>2.0.CO;2
5. Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory, 28*(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489
6. Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7
7. Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: A review and recent developments. *Philosophical Transactions of the Royal Society A, 374*(2065), 20150202. https://doi.org/10.1098/rsta.2015.0202
8. Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324
9. Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881
10. Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[utkukose.com](https://www.utkukose.com) · [ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [github.com/utkukose](https://github.com/utkukose)

<sub>NOKTA atölyesi · ÜNİDES 2026 · İlk sürüm: Kasım 2026 · İçerik CC BY 4.0, kod MIT · Veriler sentetiktir.</sub>

</div>